In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import ast
import seaborn as sns
import os

# Set aesthetically pleasing Seaborn style with larger fonts and bold text
sns.set(style="whitegrid", font_scale=1.5)
plt.rcParams['axes.labelpad'] = 12
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'

# Define three file paths
file_paths = [
    r'processed_results/allgpt/allgpt_final_with_labels.csv',
    r'processed_results/allgrok/allgrok_final_with_labels.csv',
    r'processed_results/grok-gpt/grok-gpt_final_with_labels.csv'
]

model_names = ['AllGPT', 'AllGrok', 'Grok-GPT']
feature_names = [
    'Search Cost Savings',
    'Positive Profit',
    'Privacy Loss',
    'Negative Profit',
    'Search Continuation Cost'
]

# Custom palette: Blue and Orange
palette = ['#1f77b4', '#ff7f0e']

def parse_vector(vector_str):
    try:
        return ast.literal_eval(vector_str)
    except:
        return []

def create_bar_chart(data, palette, feature_names, model_name):
    """Create a bar chart comparing features by competition intensity"""
    n_features = len(feature_names)
    n_categories = len(data)
    
    if n_categories == 0:
        print(f"Warning: No categorical data available for plotting - {model_name}")
        return
    
    # Create figure with larger size
    fig, ax = plt.subplots(figsize=(14, 9))
    
    # Calculate bar dimensions and positions
    bar_width = 0.8 / n_categories
    x_positions = np.arange(n_features)
    
    # Plot bars for each category
    for i, (label, values) in enumerate(data.iterrows()):
        # Calculate bar positions
        bar_positions = x_positions + i * bar_width - (bar_width * (n_categories - 1)) / 2
        
        # Set legend label based on competition intensity
        legend_label = 'Low-intensity competition' if label == 0 else 'High-intensity competition'
        
        # Create bars
        bars = ax.bar(bar_positions, values, bar_width * 0.9, 
                     label=legend_label, alpha=0.85, color=palette[i])
        
        # Add value labels on top of bars
        for j, value in enumerate(values):
            ax.text(bar_positions[j], value + 0.05, f'{value:.2f}', 
                   ha='center', va='bottom', fontsize=14, fontweight='bold', color='black')

    # Configure x-axis ticks and labels
    ax.set_xticks(x_positions)
    ax.set_xticklabels(feature_names, rotation=0, ha='center', fontsize=14, fontweight='bold')
    
    # Add legend
    ax.legend(title='Competition Intensity', 
              title_fontsize=16, fontsize=14, 
              prop={'weight': 'bold'}, frameon=True, loc='best')

    # Add grid
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Adjust y-axis limits
    y_min, y_max = ax.get_ylim()
    ax.set_ylim(0, y_max * 1.2)

    # Save the chart
    plt.tight_layout(pad=4.0)
    save_path = f'label2_analysis_{model_name.replace(" ", "_")}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"Bar chart saved to: {save_path}")
    return save_path

# Main processing loop
saved_files = []
for idx, file_path in enumerate(file_paths):
    model_name = model_names[idx]
    
    try:
        df = pd.read_csv(file_path)
        print(f"\n{'='*50}")
        print(f"Successfully read file: {file_path}")
        print(f"Original data row count: {len(df)}")
    except Exception as e:
        print(f"Failed to read file: {file_path}")
        print(f"Error message: {e}")
        continue
    
    # Parse vector data
    df['vector_parsed'] = df['vector_intersection'].apply(parse_vector)
    vector_lengths = df['vector_parsed'].apply(len)
    df = df[vector_lengths >= 5]
    
    # Extract features from vectors
    for i in range(5):
        df[feature_names[i]] = df['vector_parsed'].apply(lambda x: x[i] if len(x) > i else np.nan)
    
    # Remove rows with missing values
    df = df.dropna(subset=feature_names)
    
    # Check data validity
    if df.empty or 'label2' not in df.columns:
        print(f"Warning: Data is empty or missing label2 column - {file_path}")
        continue
    
    # Process label2 column
    df['label2'] = pd.to_numeric(df['label2'], errors='coerce')
    df = df.dropna(subset=['label2'])
    
    # Calculate mean values by competition intensity
    label2_means = df.groupby('label2')[feature_names].mean()
    
    if label2_means.empty:
        print(f"Warning: No valid grouped data available - {file_path}")
        continue
    
    # Create and save bar chart
    saved_file = create_bar_chart(label2_means, palette, feature_names, model_name)
    saved_files.append(saved_file)

print("\nAll bar charts generated successfully!")
print("Saved chart files:")
for file in saved_files:
    print(f"- {file}")


Successfully read file: processed_results/allgpt/allgpt_final_with_labels.csv
Original data row count: 2000
Bar chart saved to: label2_analysis_AllGPT.png

Successfully read file: processed_results/allgrok/allgrok_final_with_labels.csv
Original data row count: 2000
Bar chart saved to: label2_analysis_AllGrok.png

Successfully read file: processed_results/grok-gpt/grok-gpt_final_with_labels.csv
Original data row count: 2000
Bar chart saved to: label2_analysis_Grok-GPT.png

All bar charts generated successfully!
Saved chart files:
- label2_analysis_AllGPT.png
- label2_analysis_AllGrok.png
- label2_analysis_Grok-GPT.png
